# 3D IsoDiffusion Conditional Reconstruction

Patch-based 3D conditional diffusion over overlapping subvolumes, with the guidance and DDIM steps exposed for debugging.


## 1. Imports & Setup


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

PROJECT_ROOT = Path('/myhome/sdate')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import lovely_tensors as lt
lt.monkey_patch()

from isodiffusion.fourier_wedge import enforce_known_fourier
from isodiffusion.recon_utils import (
    iter_patch_slices,
    load_npy_volume,
    load_norm_fns_from_checkpoint_sidecar,
    load_unet3d,
    load_isonet3d,
)
from isodiffusion.schedulers.pipeline_ddim import DDIMPipeline
from isodiffusion.schedulers.scheduling_ddim import GuidedDDIMScheduler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


## 2. Configuration


In [ ]:
# Edit these paths/parameters for the current run.
condition_volume_path = Path('/myhome/data/sdate/shared/compression_paper/file_1_extracted/reconstruction/la_fourier_1.npy')
ground_truth_volume_path = Path('/myhome/data/sdate/shared/compression_paper/file_1_extracted/reconstruction/gt_fbp_1.npy')
initial_volume_path = None  # or Path('/path/to/previous_estimate.npy')
# checkpoint_path = PROJECT_ROOT / 'checkpoints' / 'ddpm_isodiffusion_isodiffusion3d.pt'
checkpoint_path = Path('/myhome/sdate/checkpoints/ddpm_isodiffusion_f1_3d.pt')
output_path = PROJECT_ROOT / 'outputs' / 'isodiffusion_recon.npy'

patch_size = 96
volume_batch_size = 6
preview_batch_index = 0

# Debug controls for the cells below.
history_every = 10
guidance_source = 'condition'  # 'ground_truth' matches the old oracle helper; use 'condition' to debug measured-data guidance.
use_identity_guidance = False

# If left as None, these are inferred from the checkpoint sidecar.
angular_range_deg = None
start_angle_deg = None
tilt_axis = None


## 3. Load Model, Normalization, and Volumes


In [ ]:
# load_unet3d automatically dispatches based on "arch" saved in the sidecar JSON.
# If the checkpoint was trained with --model_type dynunet it returns DynUNetDiffusion;
# otherwise it returns UNet3DConditionModel. No code change needed here.
model, config = load_unet3d(checkpoint_path, device=device)
normalize_fn, denormalize_fn, norm_config = load_norm_fns_from_checkpoint_sidecar(checkpoint_path)
print(f"arch: {norm_config.get('arch', 'unet3d')}")
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")
norm_config


### DynUNet alternative

Train with `--model_type dynunet` (and optionally `--dynunet_filters 32,64,128,256,320`) to use MONAI's DynUNet instead of the diffusers UNet3DConditionModel.  The checkpoint sidecar records `"arch": "dynunet"` and `load_unet3d` / `load_isonet3d` dispatch automatically — no other code changes required.

```bash
# Diffusion model with DynUNet
python isodiffusion/train_conditional_3d.py \
    --data_path /path/to/volume.npy \
    --cone_width_deg 72 \
    --model_type dynunet \
    --dynunet_filters 32,64,128,256,320 \
    --volume_size 96 --epochs 300 --exp_name my_dynunet_run

# IsoNet with DynUNet
python isodiffusion/train_isonet_3d.py \
    --data_path /path/to/volume.npy \
    --cone_width_deg 72 \
    --model_type dynunet \
    --dynunet_filters 32,64,128,256,320 \
    --volume_size 96 --epochs 300 --exp_name my_isonet_dynunet_run
```

The cell below builds a DynUNet diffusion model directly so you can inspect its architecture without a checkpoint.


In [ ]:
# Build DynUNet models directly for architecture inspection (no checkpoint needed).
from isodiffusion.dynunet_wrapper import DynUNetDiffusion, DynUNetIsoNet

filters_96 = [32, 64, 128, 256, 320]  # 5 levels → bottleneck 96/2^4 = 6^3

dynunet_diffusion = DynUNetDiffusion(in_channels=2, out_channels=1, filters=filters_96)
dynunet_isonet    = DynUNetIsoNet(in_channels=1,  out_channels=1, filters=filters_96)

print(f"DynUNetDiffusion params: {sum(p.numel() for p in dynunet_diffusion.parameters()):,}")
print(f"DynUNetIsoNet    params: {sum(p.numel() for p in dynunet_isonet.parameters()):,}")

# Smoke-test forward pass on a 96^3 volume patch.
with torch.no_grad():
    x_diff  = torch.randn(1, 2, 96, 96, 96)
    x_iso   = torch.randn(1, 1, 96, 96, 96)
    t       = torch.tensor([500])
    out_diff = dynunet_diffusion(x_diff, timestep=t, return_dict=False)[0]
    out_iso  = dynunet_isonet(x_iso, return_dict=False)[0]

print(f"DynUNetDiffusion output shape: {tuple(out_diff.shape)}")  # expect (1, 1, 96, 96, 96)
print(f"DynUNetIsoNet    output shape: {tuple(out_iso.shape)}")   # expect (1, 1, 96, 96, 96)


In [ ]:
if angular_range_deg is None:
    cone_width_deg = float(norm_config.get('cone_width_deg', 72.0))
    angular_range_deg = 180.0 - cone_width_deg
if start_angle_deg is None:
    cone_width_deg = float(norm_config.get('cone_width_deg', 72.0))
    center = float(norm_config.get('carve_center_angle_deg', 0.0))
    start_angle_deg = (center + cone_width_deg / 2.0) % 180.0
if tilt_axis is None:
    tilt_axis = int(norm_config.get('tilt_axis', 0))

# Auto-read patch_size from checkpoint config; override manually in the config cell above if needed.
_vs = norm_config.get('volume_size', patch_size)
patch_size = tuple(int(v) for v in _vs) if isinstance(_vs, list) else int(_vs)

condition = load_npy_volume(condition_volume_path)
ground_truth = load_npy_volume(ground_truth_volume_path)
initial = condition.clone() if initial_volume_path is None else load_npy_volume(initial_volume_path)

initial = ground_truth.clone()

print(f'model parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'volume shape: {tuple(condition.shape)}')
print(f'patch_size: {patch_size}')
print(f'volume batch size: {volume_batch_size}')
print(dict(angular_range_deg=angular_range_deg, start_angle_deg=start_angle_deg, tilt_axis=tilt_axis))


## 4. Crop Debug Volume


In [ ]:
# start = 400
# condition = condition[start:start + 96, start:start + 96, start:start + 96]
# initial = initial[start:start + 96, start:start + 96, start:start + 96]
# ground_truth = ground_truth[start:start + 96, start:start + 96, start:start + 96]
# condition, initial, ground_truth


condition = condition[:96]
initial = initial[:96]
ground_truth = ground_truth[:96]
condition, initial, ground_truth


## 5. Patch Geometry

`reconstruct_volume_patches` builds this list internally. Keeping it visible makes it easy to inspect the exact patch batch that enters the denoiser.


In [ ]:
overlap = 10
patch_slices = list(iter_patch_slices(condition.shape[-3:], patch_size, overlap))
debug_patch_index = 20
debug_patch_index = min(debug_patch_index, len(patch_slices) - 1)
debug_slc = patch_slices[debug_patch_index]

def make_volume_batch(volume: torch.Tensor, batch_size: int) -> torch.Tensor:
    if volume.ndim == 3:
        return volume.unsqueeze(0).repeat(batch_size, 1, 1, 1).contiguous()
    if volume.ndim == 4:
        if volume.shape[0] != batch_size:
            raise ValueError(f'batched volume has batch size {volume.shape[0]}, expected {batch_size}')
        return volume.contiguous()
    raise ValueError(f'expected (D, H, W) or (bs, D, H, W), got {tuple(volume.shape)}')

condition_batch = make_volume_batch(condition, volume_batch_size)
initial_batch = make_volume_batch(initial, volume_batch_size)
ground_truth_batch = make_volume_batch(ground_truth, volume_batch_size)

debug_initial = initial_batch[:, debug_slc[0], debug_slc[1], debug_slc[2]]
debug_condition = condition_batch[:, debug_slc[0], debug_slc[1], debug_slc[2]]
debug_ground_truth = ground_truth_batch[:, debug_slc[0], debug_slc[1], debug_slc[2]]

print(f'patches per volume: {len(patch_slices)}')
print(f'total packed subvolumes: {volume_batch_size * len(patch_slices)}')
print(f'debug patch {debug_patch_index}: {debug_slc}')
print(f'debug patch batch shape: {tuple(debug_condition.shape)}')


## 6. Full-Volume Guidance Function

The new `DDIMPipeline` partitions only the UNet call into 3D subvolumes. The scheduler keeps the raw state as `(bs, D, H, W)`, so `guidance(x_0_raw, t)` receives and returns the same full-volume batch shape.


In [ ]:
def select_guidance_volume(condition_volume: torch.Tensor, ground_truth_volume: torch.Tensor) -> torch.Tensor:
    if guidance_source == 'ground_truth':
        return ground_truth_volume
    if guidance_source == 'condition':
        return condition_volume
    raise ValueError("guidance_source must be 'ground_truth' or 'condition'")


def fourier_guidance(
    source_volume: torch.Tensor,
    vol_init: torch.Tensor,
    timestep: int = 0,
) -> torch.Tensor:
    if vol_init.ndim == 3:
        if source_volume.ndim == 4:
            source_volume = source_volume[0]
        return enforce_known_fourier(
            estimate_volume=vol_init,
            measured_volume=source_volume.to(vol_init.device, dtype=vol_init.dtype),
            angular_range_deg=angular_range_deg,
            start_angle_deg=start_angle_deg,
            tilt_axis=tilt_axis,
        )

    if vol_init.ndim != 4:
        raise ValueError(f'guidance expects (D, H, W) or (bs, D, H, W), got {tuple(vol_init.shape)}')

    if source_volume.ndim == 3:
        source_volume = source_volume.unsqueeze(0).expand(vol_init.shape[0], -1, -1, -1)
    if source_volume.ndim != 4 or source_volume.shape[0] != vol_init.shape[0]:
        raise ValueError(
            f'source volume shape {tuple(source_volume.shape)} is incompatible with estimate shape {tuple(vol_init.shape)}'
        )

    guided = [
        enforce_known_fourier(
            estimate_volume=estimate,
            measured_volume=measured.to(estimate.device, dtype=estimate.dtype),
            angular_range_deg=angular_range_deg,
            start_angle_deg=start_angle_deg,
            tilt_axis=tilt_axis,
        )
        for estimate, measured in zip(vol_init, source_volume)
    ]
    return torch.stack(guided, dim=0)


guidance_source_volume = select_guidance_volume(condition_batch, ground_truth_batch)

def guidance(x_0_raw: torch.Tensor, timestep: int) -> torch.Tensor:
    if use_identity_guidance:
        return x_0_raw
    x = fourier_guidance(
        source_volume=guidance_source_volume,
        vol_init=x_0_raw,
        timestep=int(timestep),
    )
    x = torch.mean(x, dim=0).repeat(x.shape[0], 1, 1)  # average over slices to get a single slice (for testing)
    return x

print(f'guidance source: {guidance_source}')
print(f'guidance batch shape: {tuple(guidance_source_volume.shape)}')


In [ ]:
# Quick guidance sanity check from zero initialisation on the full crop batch.
with torch.no_grad():
    zeros_raw = torch.zeros_like(guidance_source_volume)
    guided_from_zeros = guidance(zeros_raw.to(device), 0).cpu()

preview_idx = min(preview_batch_index, guided_from_zeros.shape[0] - 1)
guidance_preview = guidance_source_volume[preview_idx].cpu()
condition_preview = condition_batch[preview_idx].cpu()
guided_preview = guided_from_zeros[preview_idx]

z = guidance_preview.shape[0] // 2
vmin, vmax = np.percentile(guidance_source_volume.numpy(), [1, 99])

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(guidance_preview[z], cmap='gray', vmin=vmin, vmax=vmax)
axes[0].set_title('guidance source')
axes[1].imshow(condition_preview[z], cmap='gray', vmin=vmin, vmax=vmax)
axes[1].set_title('condition')
axes[2].imshow(guided_preview, cmap='gray', vmin=vmin, vmax=vmax)
axes[2].set_title('guided from zeros')
axes[3].imshow((guided_preview - guidance_preview[z]).abs(), cmap='magma')
axes[3].set_title('|guided - source|')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
torch.cuda.empty_cache()


In [ ]:
torch.cuda.empty_cache()
NUM_INFERENCE_STEPS = 50
scheduler = GuidedDDIMScheduler(1000, clip_sample_range=6)
scheduler.set_timesteps(NUM_INFERENCE_STEPS)
timestep = 300
x = normalize_fn(debug_ground_truth[:1])
conditioning = normalize_fn(debug_condition[:1].squeeze())
x_t = scheduler.add_noise(x, torch.randn_like(x), timesteps=torch.tensor([timestep])).to(device)
with torch.no_grad():
    model_input = torch.stack([x_t[0], conditioning.to(device)], dim=0).to(device)  # (2, H, W)
    
    encoder_hidden_states = torch.zeros(
        1,
        1,
        int(model.config.cross_attention_dim),
        device=device,
        dtype=model.dtype,
    )

    noise_pred = model(model_input.unsqueeze(0), timestep = torch.tensor([timestep]).to(device), encoder_hidden_states=encoder_hidden_states, return_dict=False)[0].squeeze()  # (H, W)
    x_0_pred = scheduler.step(noise_pred, timestep=timestep, sample=x_t)['pred_original_sample']  # (H, W)
    
    diffusion_pred = denormalize_fn(x_0_pred)
    # diffusion_pred = inpaint_guidance(
    #     source_volume=gt_slice.unsqueeze(0).to(device),              # oracle: original full-angle tensor on device
    #     vol_init=diffusion_pred.unsqueeze(0).to(device)
    # )
z = diffusion_pred.shape[1] // 2
fig, axes = plt.subplots(1, 4, figsize=(18, 6))
axes[0].imshow(diffusion_pred.squeeze().cpu()[z], cmap='gray', vmin=vmin, vmax=vmax)
axes[0].set_title('diffusion prediction')
axes[1].imshow(debug_condition[:1].squeeze()[z], cmap='gray', vmin=vmin, vmax=vmax)
axes[1].set_title('condition')
axes[2].imshow(debug_ground_truth[:1].squeeze()[z], cmap='gray', vmin=vmin, vmax=vmax)
axes[2].set_title('guidance source')
axes[3].imshow(x_t.squeeze().cpu()[z], cmap='gray')
axes[3].set_title('x_t (noisy input)')
x_t

## 7. Run Guided DDIM Pipeline

This mirrors `test_large_ladiff_conditional`: create a `GuidedDDIMScheduler`, pass it into `DDIMPipeline`, then call `truncated_pipeline(...)` on the current full-volume estimate.


In [ ]:
overlap = 10
subvolume_batch_size = 3
num_inference_steps = 20
start_step_frac = 0.1
num_outer_iters = 1


In [ ]:
# guidance = lambda x_0, t: x_0  # identity guidance for testing the rest of the pipeline without the Fourier constraint.

In [ ]:
torch.cuda.empty_cache()

noise_scheduler = GuidedDDIMScheduler(
    num_train_timesteps=1000,
    guidance_function=guidance,
    clip_sample_range=6
)

pipeline = DDIMPipeline(
    unet=model.to(device),
    scheduler=noise_scheduler,
    conditioning=condition_batch.to(device),
    normalize_fn=normalize_fn,
    denormalize_fn=denormalize_fn,
    subvolume_batch_size=subvolume_batch_size,
    overlap=overlap,
)

recon = initial_batch.float()

result = pipeline.truncated_pipeline(
    initial_guess=recon.to(device),
    start_step=int(start_step_frac * num_inference_steps),
    num_inference_steps=num_inference_steps,
    use_clipped_model_output=True,
    p_use_conditioning=1.0,
)
recon = result.images

# make sure the reconstruction has all the fourier features
recon = guidance(recon.to(device), 0).cpu()

output_path.parent.mkdir(parents=True, exist_ok=True)
np.save(output_path, recon.numpy().astype(np.float32))

print(f'Guided DDIM reconstruction done')
print(f'  shape: {tuple(recon.shape)}')
print(f'  range: [{recon.min():.4f}, {recon.max():.4f}]')
output_path


## 8. Visualise Result


In [ ]:
preview_idx = min(preview_batch_index, recon.shape[0] - 1)
recon_preview = recon[preview_idx]
condition_preview = condition_batch[preview_idx]
ground_truth_preview = ground_truth_batch[preview_idx]

z = recon_preview.shape[0] // 2
z = 0
vmin, vmax = np.percentile(condition_preview.numpy(), [1, 99])

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(condition_preview[z], cmap='gray', vmin=vmin, vmax=vmax)
axes[0].set_title('condition')
axes[1].imshow(recon_preview[z], cmap='gray', vmin=vmin, vmax=vmax)
axes[1].set_title('isodiffusion')
axes[2].imshow(ground_truth_preview[z], cmap='gray', vmin=vmin, vmax=vmax)
axes[2].set_title('ground truth')
axes[3].imshow((recon_preview[z] - ground_truth_preview[z]).abs(), cmap='magma')
axes[3].set_title('|error|')
for ax in axes:
    ax.axis('off')
plt.tight_layout()


In [ ]:
from ipywidgets import interact
import ipywidgets as widgets

preview_idx = min(preview_batch_index, recon.shape[0] - 1)
recon_preview = recon[preview_idx]
condition_preview = condition_batch[preview_idx]
ground_truth_preview = ground_truth_batch[preview_idx]

vmin, vmax = np.percentile(condition_preview.numpy(), [1, 99])

def visualize_slice(z):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].imshow(condition_preview[z], cmap='gray', vmin=vmin, vmax=vmax)
    axes[0].set_title('condition')
    axes[1].imshow(recon_preview[z], cmap='gray', vmin=vmin, vmax=vmax)
    axes[1].set_title('isodiffusion')
    axes[2].imshow(ground_truth_preview[z], cmap='gray', vmin=vmin, vmax=vmax)
    axes[2].set_title('ground truth')
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

max_z = recon_preview.shape[0] - 1
interact(visualize_slice, z=widgets.IntSlider(min=0, max=max_z, step=1, value=max_z//2))